# Intersections and wall thickness: `find_intersections` and `thickness`

Two checks on a surface before it is meshed, printed, cast or assembled (roadmap 3.1.2 and 3.1.4):

| operation | asks | answers with |
|---|---|---|
| `find_intersections(mesh)` | does the surface cross itself? | the intersecting cell pairs, the crossing segments as a line mesh, `intersections:count` per cell |
| `find_intersections(a, b)` | do two parts interpenetrate or touch? | the same, one count array per part |
| `thickness(mesh)` | how thick is the wall at each facet? | `thickness:distance` by a ray against the facet's normal to the far side |
| `thickness(a, b)` | how far is each facet of `a` from `b`? | the clearance by a ray along the normal |

Every intersection is decided by an exact triangle–triangle test on Shewchuk's `orient3d`/`orient2d`, so there is no tolerance and no list of "possible" hits.

In [ ]:
import os
import subprocess
import sys
import tempfile

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection

import meshioplusplus as mp


def gl_available():
    # Can PyVista render off-screen here? VTK without a GL/EGL/OSMesa library aborts
    # the process instead of raising, so the probe runs in a child process.
    probe = ("import pyvista as pv; pv.OFF_SCREEN = True; "
             "pl = pv.Plotter(off_screen=True, window_size=(64, 64)); "
             "pl.add_mesh(pv.Sphere()); pl.screenshot(return_img=True)")
    try:
        return subprocess.run([sys.executable, '-c', probe], capture_output=True, timeout=120).returncode == 0
    except Exception:
        return False


try:
    import pyvista as pv

    if gl_available():
        # Head-less, static rendering (no display / live server needed).
        pv.OFF_SCREEN = True
        pv.set_jupyter_backend('static')
    else:
        pv = None
        print('No GL available: rendering with matplotlib.')
except ImportError:  # pragma: no cover - pyvista is optional here
    pv = None
    print('PyVista unavailable: rendering with matplotlib.')


def show(img, title=None, figsize=(6, 4.5)):
    """Embed a rendered RGB screenshot."""
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis('off')
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.show()


def to_pyvista(mesh):
    """meshio++ Mesh -> pyvista grid via a temporary .vtu."""
    with tempfile.NamedTemporaryFile(suffix='.vtu', delete=False) as f:
        tmp = f.name
    try:
        mp.write(tmp, mesh)
        return pv.read(tmp)
    finally:
        os.unlink(tmp)


SURFACE_TYPES = {'triangle': 3, 'quad': 4}


def faces_and_values(mesh, array):
    """Corner polygons of the 2-D cells and, per polygon, the cell value of `array`."""
    polys, values = [], []
    for b, block in enumerate(mesh.cells):
        if block.type not in SURFACE_TYPES:
            continue
        data = np.asarray(block.data)
        vals = np.asarray(mesh.cell_data[array][b]) if array else np.zeros(len(data))
        polys += [mesh.points[row[:SURFACE_TYPES[block.type]]] for row in data]
        values += list(vals)
    return polys, np.asarray(values, dtype=float)


def render(meshes, title, scalars=None, cmap='viridis', lines=None, clim=None, figsize=(6, 4.5)):
    """PyVista off-screen render embedded through matplotlib; falls back to matplotlib alone.

    `meshes` is one mesh or a list; `scalars` names a cell array to colour by (NaN grey);
    `lines` is a line mesh drawn over them in red."""
    meshes = meshes if isinstance(meshes, list) else [meshes]
    if pv is not None:
        try:
            pl = pv.Plotter(off_screen=True, window_size=(900, 650))
            for m in meshes:
                grid = to_pyvista(m)
                if scalars:
                    pl.add_mesh(grid, scalars=scalars, cmap=cmap, clim=clim, nan_color='#cccccc',
                                show_edges=True, edge_color='#1f4e79', line_width=0.2)
                else:
                    pl.add_mesh(grid, color='#2ec4b6', show_edges=True, edge_color='#1f4e79', line_width=0.2)
            if lines is not None and len(lines.points):
                pl.add_mesh(to_pyvista(lines), color='#e63946', line_width=5)
            pl.camera_position = 'iso'
            img = pl.screenshot(return_img=True)
            pl.close()
            show(img, title, figsize=figsize)
            return
        except Exception as exc:  # head-less GL fallback
            print('PyVista render skipped, drawing with matplotlib:', type(exc).__name__)
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection='3d')
    polys, vals = [], []
    for m in meshes:
        p, v = faces_and_values(m, scalars)
        polys += p
        vals += list(v)
    vals = np.asarray(vals)
    if scalars:
        finite = vals[np.isfinite(vals)]
        lo, hi = clim if clim else ((finite.min(), finite.max()) if finite.size else (0.0, 1.0))
        norm = colors.Normalize(lo, hi if hi > lo else lo + 1.0)
        face = [cm.get_cmap(cmap)(norm(v)) if np.isfinite(v) else (0.8, 0.8, 0.8, 1.0) for v in vals]
        fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.6, label=scalars)
    else:
        face = '#2ec4b6'
    ax.add_collection3d(Poly3DCollection(polys, facecolor=face, edgecolor='#1f4e79', linewidths=0.2, alpha=0.9))
    if lines is not None and len(lines.points):
        segs = lines.points[np.asarray(lines.cells[0].data)]
        ax.add_collection3d(Line3DCollection(segs, colors='#e63946', linewidths=2.5))
    pts = np.concatenate(polys)
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    centre, half = (lo + hi) / 2, (hi - lo).max() / 2
    ax.set_xlim(centre[0] - half, centre[0] + half)
    ax.set_ylim(centre[1] - half, centre[1] + half)
    ax.set_zlim(centre[2] - half, centre[2] + half)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    ax.view_init(elev=22, azim=-55)
    ax.set_title(title)
    plt.show()


def flipped(mesh):
    """The surface wound the other way round (its normals pointing inward)."""
    return mp.Mesh(mesh.points, [(b.type, np.asarray(b.data)[:, ::-1]) for b in mesh.cells])

## Two parts that interpenetrate

The roadmap's probe: two unit spheres one radius apart. Against a second surface every contact counts, touching included. The pairs come back as global cell ids, the crossing segments as a line mesh, and each part carries `intersections:count` — how many cells of the other part each of its cells meets.

In [ ]:
a = mp.icosa_sphere(1.0, 3)
b = mp.icosa_sphere(1.0, 3, center=(1.0, 0.0, 0.0))
flag_a, rep = mp.find_intersections(a, b, return_report=True)
print({k: rep[k] for k in ('num_pairs', 'num_triangle_pairs', 'num_coplanar', 'num_segments', 'num_cells_a', 'num_cells_b', 'clean')})
print('first pairs:', rep['pairs'][:4].tolist())

seg = rep['segments']
ends = seg.points
length = np.linalg.norm(ends[1::2] - ends[0::2], axis=1).sum()
print(f'segments on x = 0.5: max |x - 0.5| = {np.abs(ends[:, 0] - 0.5).max():.2e}')
print(f'loop length {length:.4f} vs the circle 2 pi sqrt(0.75) = {2 * np.pi * np.sqrt(0.75):.4f}')
render([flag_a, rep['other']], 'two spheres: intersections:count and the crossing segments',
       scalars='intersections:count', cmap='plasma', lines=seg)

## A surface that crosses itself

On one surface, neighbours that share corners are not an intersection — only meeting *beyond* what they share is. Corners count as shared when their coordinates are bitwise equal, so an unwelded seam (here every triangle with points of its own) is still clean. Pushing a cap of the sphere through the far side makes it cross itself.

In [ ]:
s = mp.icosa_sphere(1.0, 3)
_, clean = mp.find_intersections(s, return_report=True)
tri = np.asarray(s.cells[0].data)
seams = mp.Mesh(s.points[tri.ravel()], [('triangle', np.arange(tri.size).reshape(-1, 3))])
print('sphere clean:', clean['clean'], '| unwelded at every edge, clean:', mp.find_intersections(seams, return_report=True)[1]['clean'])

pushed = s.points.copy()
cap = pushed[:, 0] > 0.6
pushed[cap, 0] -= 1.9 * (pushed[cap, 0] - 0.6) / 0.4
dented = mp.Mesh(pushed, s.cells)
flags, rep = mp.find_intersections(dented, return_report=True)
print({k: rep[k] for k in ('num_pairs', 'num_segments', 'num_cells_a', 'clean')})
render(flags, 'a cap pushed through the far side: the cells that cross',
       scalars='intersections:count', cmap='plasma', lines=rep['segments'])

## Wall thickness

`thickness(mesh)` casts a ray from each facet's centroid against its outward normal and stops at the first wall it leaves through: the far side of the wall. Here a sphere of radius 1 holds a cavity of radius 0.6 pushed off-centre by 0.25, so the wall runs from 0.15 to 0.65; the cavity's surface is wound with its normals into the cavity, out of the solid, as a consistently oriented shell is.

In [ ]:
outer = mp.icosa_sphere(1.0, 4)
cavity = flipped(mp.icosa_sphere(0.6, 4, center=(0.25, 0.0, 0.0)))
shell = mp.merge([outer, cavity])
measured, rep = mp.thickness(shell, threshold=0.25, return_report=True)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rep.items() if k not in ('min_point', 'warnings')})
print('thinnest wall at', np.round(rep['min_point'], 3))
n_outer = len(outer.cells[0].data)
skin = mp.Mesh(outer.points, outer.cells,
               cell_data={'thickness:distance': [np.concatenate(measured.cell_data['thickness:distance'])[:n_outer]]})
render(skin, 'wall thickness on the outer skin (thinnest at +x)', scalars='thickness:distance', cmap='viridis')

## Clearance between two parts

Against a second surface the ray runs along the outward normal and stops at the first wall facing it, so `thickness(a, b)` reads the gap from each facet of `a` to `b`; rays that leave into open space are misses (NaN, grey).

In [ ]:
a = mp.icosa_sphere(1.0, 3)
b = mp.icosa_sphere(0.7, 3, center=(2.0, 0.6, 0.0))
gap_mesh, gap = mp.thickness(a, b, return_report=True)
print(f"clearance {gap['min']:.4f} at {np.round(gap['min_point'], 3)}; {gap['num_hit']} facets see b, {gap['num_missed']} do not")
print('the exact sphere-to-sphere gap:', round(np.hypot(2.0, 0.6) - 1.7, 4))
render([gap_mesh, b], 'clearance from a to b along each facet\'s normal', scalars='thickness:distance', cmap='magma_r')

## On the command line and in a pipeline

Both CLIs carry the two verbs with the same `--json` shape: `find-intersections` exits 1 when anything intersects (2 when it cannot check), and `thickness --min T` exits 1 when a facet is thinner than `T`. In a pipeline `FindIntersections` is a gate like `CheckConformity`, and `Thickness` with `Min` stops a mesh that is too thin.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'shell.vtu')
    mp.write(path, shell)
    report = mp.run_pipeline({
        'Version': 1,
        'Input': {'Path': path},
        'Operations': [
            {'Op': 'FindIntersections'},
            {'Op': 'Thickness', 'Min': 0.1},
        ],
        'Output': {'Path': os.path.join(d, 'checked.vtu')},
    })
for step in report['steps']:
    print(step)